# ADL Results Explorer

Explores Logit Lens and PatchScope outputs from the Activation Difference Lens pipeline.

In [2]:
from pathlib import Path
import matplotlib.pyplot as plt

# --- Configuration (edit these) ---
RESULTS_DIR = Path(
    "../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_narrow-sft-2/activation_difference_lens"
)
LAYERS = [7, 14, 15]
DATASET = "tulu-3-sft-olmo-2-mixture"
LOGIT_LENS_POSITION = -1  # Position for per-position logit lens view
PATCHSCOPE_POSITION = -1  # Position for per-position patchscope view
N_POSITIONS = 128  # Total positions (config: n)
LOGIT_LENS_MAX_ROWS = 20  # Set to an integer to truncate logit lens tables
PATCHSCOPE_GRADER = "openai_gpt-5-mini"
MODEL_ID = "allenai/OLMo-2-0425-1B-DPO"

LAYER_DIRS = {layer: RESULTS_DIR / f"layer_{layer}" / DATASET for layer in LAYERS}

In [3]:
import re
import torch
import pandas as pd
from collections import defaultdict
from transformers import AutoTokenizer

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 60)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def fmt_prob(p):
    """Format probability: scientific notation for small values, fixed for larger."""
    if abs(p) < 0.01:
        return f"{p:.2e}"
    return f"{p:.4f}"


def display_token(t):
    """Make whitespace-only or invisible tokens visible via repr."""
    if not t.strip():
        return repr(t)
    return t


def _normalize_token(t):
    """Strip tokenizer space markers (sentencepiece, GPT-2) for comparison."""
    return t.replace("\u2581", "").replace("\u0120", "").strip()


def load_logit_lens(layer, pos, prefix=""):
    """Load logit lens .pt file. Returns (top_k_probs, top_k_indices, inv_probs, inv_indices)."""
    return torch.load(
        LAYER_DIRS[layer] / f"{prefix}logit_lens_pos_{pos}.pt", weights_only=True
    )


def decode_tokens(indices):
    return [tokenizer.decode([int(i)]) for i in indices]


def load_patchscope(layer, pos, prefix=""):
    """Load auto_patch_scope .pt file. Returns dict with tokens_at_best_scale, selected_tokens, etc."""
    return torch.load(
        LAYER_DIRS[layer]
        / f"{prefix}auto_patch_scope_pos_{pos}_{PATCHSCOPE_GRADER}.pt",
        weights_only=False,
    )


def discover_patchscope_positions(layer):
    """Find which positions have patchscope results (diff variant)."""
    positions = []
    for f in sorted(
        LAYER_DIRS[layer].glob(f"auto_patch_scope_pos_*_{PATCHSCOPE_GRADER}.pt")
    ):
        m = re.search(r"auto_patch_scope_pos_(\d+)_", f.name)
        if m:
            positions.append(int(m.group(1)))
    return positions


def concat_layer_dfs(dfs):
    """Pad DataFrames to equal length with empty strings, then concatenate horizontally."""
    max_len = max(len(df) for df in dfs)
    padded = []
    for df in dfs:
        if len(df) < max_len:
            pad = pd.DataFrame(
                {col: [""] * (max_len - len(df)) for col in df.columns},
                index=range(len(df), max_len),
            )
            df = pd.concat([df, pad], axis=0)
        padded.append(df)
    return pd.concat(padded, axis=1)


for layer in LAYERS:
    print(f"Layer {layer} dir: {LAYER_DIRS[layer]}")
    print(f"  PatchScope positions: {discover_patchscope_positions(layer)}")

Layer 7 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_narrow-sft-2/activation_difference_lens/layer_7/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 14 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_narrow-sft-2/activation_difference_lens/layer_14/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]
Layer 15 dir: ../../../adl_results/workspace/model-organisms/diffing_results/olmo2_1B/examples_narrow-sft-2/activation_difference_lens/layer_15/tulu-3-sft-olmo-2-mixture
  PatchScope positions: [0, 1, 2, 3, 4, 5]


## 1. Logit Lens Analysis

### 1A. Single Position

Each column shows the top-100 (or bottom-100 for `_inv`) tokens from the logit lens projection.  
Format: `token (softmax_prob)`

In [4]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {LOGIT_LENS_POSITION}:")
logit_lens_position_table(LOGIT_LENS_POSITION)

Logit lens at position -1:


layer_7                                                    \
                       base             base_inv                           ft   
0           .Today (0.0249)      urrenc (0.0210)               oller (0.0216)   
1          .Second (0.0110)       act (5.13e-03)           Buccane (6.81e-03)   
2        Buccane (8.61e-03)       pos (4.97e-03)          concrete (4.82e-03)   
3          /Area (5.92e-03)    askell (3.52e-03)              .Abs (4.00e-03)   
4            .au (4.76e-03)      gons (3.11e-03)              fern (3.75e-03)   
5            aru (3.71e-03)        �� (2.01e-03)           current (3.31e-03)   
6      /problems (3.71e-03)      ejec (2.01e-03)         generally (2.75e-03)   
7      /entities (2.98e-03)      anth (1.95e-03)              iger (2.21e-03)   
8          /bind (2.62e-03)      azon (1.89e-03)            .There (2.14e-03)   
9          /Math (2.62e-03)        دي (1.89e-03)         according (2.08e-03)   
10      /respond (2.55e-03)     fácil (1.83e-03)        engagement (1.95e-03)   
11      /problem (2.55e-03)     posix (1.72e-03)           studies (1.95e-03)   
12          fter (2.47e-03)  essional (1.67e-03)       sovereignty (1.95e-03)   
13     /operator (2.24e-03)      Vers (1.67e-03)               cly (1.95e-03)   
14     /activity (2.18e-03)  Optional (1.52e-03)             isión (1.77e-03)   
15    confidence (2.11e-03)    Phones (1.47e-03)         depending (1.56e-03)   
16   persistence (2.11e-03)  >Welcome (1.43e-03)             would (1.56e-03)   
17     belonging (1.98e-03)       med (1.38e-03)               riv (1.52e-03)   
18          ilot (1.92e-03)        vs (1.30e-03)   notwithstanding (1.47e-03)   
19     .AddRange (1.80e-03)      orst (1.26e-03)              ream (1.47e-03)   

                                                                         \
                ft_inv                    diff                 diff_inv   
0        spos (0.0112)            ach (0.0129)           ource (0.0688)   
1   ategori (7.69e-03)          ' \n' (0.0118)             neh (0.0105)   
2    urrenc (6.77e-03)        domin (7.17e-03)           ICA (5.98e-03)   
3       pos (6.38e-03)          mon (7.17e-03)          iore (4.97e-03)   
4       pon (4.12e-03)        Abyss (5.95e-03)         rosse (3.88e-03)   
5     aland (2.66e-03)            次 (3.72e-03)   cellspacing (3.63e-03)   
6     suger (2.49e-03)            @ (3.60e-03)          ouch (3.02e-03)   
7     carga (2.49e-03)            ( (3.17e-03)         AMILY (3.02e-03)   
8   ponsors (2.20e-03)            � (2.99e-03)           eto (2.84e-03)   
9    istema (2.06e-03)   influences (2.72e-03)            яз (2.66e-03)   
10     acja (1.94e-03)            O (2.64e-03)       ---\n\n (2.66e-03)   
11    fácil (1.94e-03)       master (2.64e-03)           asu (2.35e-03)   
12     culo (1.82e-03)         Gold (2.55e-03)         /tags (2.35e-03)   
13    abril (1.82e-03)            d (2.55e-03)        istema (2.21e-03)   
14     pons (1.72e-03)      supreme (2.47e-03)        /lists (2.21e-03)   
15     THEN (1.72e-03)      pressed (2.47e-03)         marzo (2.08e-03)   
16   askell (1.72e-03)     imestamp (2.29e-03)        champs (1.95e-03)   
17    mostr (1.72e-03)        -eyed (2.26e-03)      <section (1.95e-03)   
18      agi (1.72e-03)        oller (2.01e-03)           ioc (1.95e-03)   
19      era (1.61e-03)          sea (1.95e-03)           kao (1.83e-03)   

                layer_14                                           \
                    base               base_inv                ft   
0            To (0.9219)          zoek (0.8516)        1 (0.3242)   
1           The (0.0430)      contador (0.1309)      The (0.3047)   
2           Let (0.0109)             메 (0.0107)        I (0.1436)   
3            In (0.0103)         иск (3.94e-03)       To (0.0601)   
4         ### (3.77e-03)     Produto (2.40e-03)       As (0.0530)   
5           A (2.29e-03)           � (1.42e-05)       In (0.0342)   
6          As (1.30e-03)      Rese

In [5]:
# Logit lens columns: (file prefix, tuple index for probs, tuple index for indices)
LL_VARIANTS = {
    "base": ("base_", 0, 1),
    "base_inv": ("base_", 2, 3),
    "ft": ("ft_", 0, 1),
    "ft_inv": ("ft_", 2, 3),
    "diff": ("", 0, 1),
    "diff_inv": ("", 2, 3),
}


def logit_lens_position_table_single(layer, pos):
    cols = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        data = load_logit_lens(layer, pos, prefix)
        tokens = decode_tokens(data[ii])
        probs = data[pi].tolist()
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)
        ]
    df = pd.DataFrame(cols)
    if LOGIT_LENS_MAX_ROWS is not None:
        df = df.head(LOGIT_LENS_MAX_ROWS)
    return df


def logit_lens_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = logit_lens_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"Logit lens at position {5}:")
logit_lens_position_table(5)

Logit lens at position 5:


layer_7                                                \
                       base             base_inv                       ft   
0         /problem (0.0403)       .vn (7.72e-03)        /problem (0.0198)   
1        /entities (0.0286)       (us (4.97e-03)       /entities (0.0128)   
2        /problems (0.0173)       ]]; (4.15e-03)       /problems (0.0116)   
3           .Today (0.0102)      sagt (3.89e-03)          '\n' (5.31e-03)   
4        /global (8.73e-03)        że (3.65e-03)       /manage (4.85e-03)   
5           /job (7.69e-03)    testim (3.02e-03)        .Round (4.27e-03)   
6        /manage (7.69e-03)      -ves (2.84e-03)       /global (4.27e-03)   
7   /preferences (6.38e-03)       ')" (2.84e-03)         scarc (3.66e-03)   
8        /layout (5.98e-03)       ($. (2.84e-03)       /layout (3.43e-03)   
9      /provider (5.13e-03)     zeigt (2.67e-03)      /logging (3.33e-03)   
10       /crypto (4.79e-03)     feliz (2.35e-03)        .Today (3.23e-03)   
11   /connection (4.52e-03)     spons (2.21e-03)         :\n\n (3.23e-03)   
12      /logging (4.24e-03)     lesen (2.08e-03)  /environment (3.04e-03)   
13    WHATSOEVER (4.00e-03)   kontrol (1.95e-03)      /account (3.04e-03)   
14       /engine (3.86e-03)       (!! (1.95e-03)     /provider (2.59e-03)   
15        .Round (3.74e-03)      helf (1.72e-03)   /categories (2.55e-03)   
16          /reg (3.63e-03)    spiele (1.72e-03)   /connection (2.37e-03)   
17  /environment (3.63e-03)     scrut (1.62e-03)        /basic (2.32e-03)   
18       /dialog (3.40e-03)       )": (1.52e-03)        /legal (2.29e-03)   
19       /entity (3.40e-03)       fas (1.43e-03)          /job (2.15e-03)   

                                                                            \
                   ft_inv                   diff                  diff_inv   
0            .vn (0.0113)             — (0.0898)           anmeld (0.0150)   
1           że (7.29e-03)         ' \n' (0.0273)               aż (0.0103)   
2          ($. (6.44e-03)    Contract (9.46e-03)           icap (6.62e-03)   
3        spons (4.70e-03)       Dimit (7.35e-03)            neh (5.86e-03)   
4          (!! (4.70e-03)          Pe (4.91e-03)       perience (5.16e-03)   
5         -ves (4.15e-03)   contracts (4.76e-03)             zł (4.85e-03)   
6        scrut (4.15e-03)        obao (4.46e-03)            ncy (4.03e-03)   
7         helf (3.91e-03)    contract (3.37e-03)  "\r\n\r\n\r\n (3.78e-03)   
8       testim (3.04e-03)        fran (3.16e-03)            iez (3.78e-03)   
9     ,,,,,,,, (3.04e-03)         PPP (3.07e-03)            aes (2.94e-03)   
10   possibile (3.04e-03)     ' \n\n' (3.07e-03)             ?v (2.94e-03)   
11        sagt (3.04e-03)    contract (2.88e-03)            iei (2.76e-03)   
12     personn (2.85e-03)    lections (2.62e-03)            >[] (2.44e-03)   
13         -ie (2.85e-03)        whom (2.55e-03)            fic (2.29e-03)   
14       asign (2.69e-03)         kel (2.47e-03)     -translate (1.90e-03)   
15         bmi (2.37e-03)       bonds (2.32e-03)            ick (1.79e-03)   
16       zeigt (2.37e-03)        '  ' (2.11e-03)            jit (1.39e-03)   
17    protagon (2.37e-03)         vim (1.92e-03)            <<= (1.39e-03)   
18       lesen (2.09e-03)        pond (1.80e-03)    >\n\n\n\n\n (1.39e-03)   
19         )": (1.97e-03)        able (1.69e-03)          -pill (1.39e-03)   

            layer_14                                           \
                base              base_inv                 ft   
0         , (0.5664)     contador (0.8398)         , (0.4863)   
1       the (0.1494)         bö (7.26e-03)       ' ' (0.1631)   
2       and (0.1387)    kontrol (7.26e-03)       the (0.1118)   
3        in (0.0615)   karakter (7.26e-03)       and (0.1084)   
4       ' ' (0.0491)         �� (5.68e-03)        in (0.0771)   
5         a (0.0139)         �� (5.68e-03)         a (0.0164)   
6      to (3.69e-03)      subur (3.43e-03)      to (4.91e-03)   
7      of (3.56e-03)   

### 1B. Aggregated Across All Positions

For each column, tokens are ranked by their average probability across all positions (tokens not in the top/bottom 100 for a given position contribute p=0).  
Format: `token (avg_prob)`

In [6]:
def logit_lens_aggregated_single(layer):
    agg = {}
    for col_name, (prefix, pi, ii) in LL_VARIANTS.items():
        token_prob_sum = defaultdict(float)
        for pos in range(N_POSITIONS):
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
        token_avg = {t: s / N_POSITIONS for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        limit = LOGIT_LENS_MAX_ROWS if LOGIT_LENS_MAX_ROWS is not None else 100
        agg[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            for t in sorted_tokens[:limit]
        ]

    max_len = max(len(v) for v in agg.values())
    for k in agg:
        agg[k] += [""] * (max_len - len(agg[k]))
    return pd.DataFrame(agg)


def logit_lens_aggregated():
    dfs = []
    for layer in LAYERS:
        df = logit_lens_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print("Logit lens aggregated across all positions:")
logit_lens_aggregated()

Logit lens aggregated across all positions:


layer_7                                                \
                       base             base_inv                       ft   
0        /entities (0.0272)         .vn (0.0199)        /problem (0.0119)   
1         /problem (0.0142)   /Register (0.0114)       /entities (0.0118)   
2      /problems (9.26e-03)    testim (6.91e-03)       /problems (0.0117)   
3        /global (6.64e-03)      sagt (6.46e-03)             , (5.73e-03)   
4         .Today (6.06e-03)     asign (5.23e-03)     /provider (5.29e-03)   
5      /provider (5.89e-03)       -ie (4.92e-03)   /connection (4.72e-03)   
6   /environment (5.85e-03)     zeigt (4.07e-03)      /account (4.29e-03)   
7    /connection (5.77e-03)        że (3.94e-03)       /global (3.66e-03)   
8        /manage (5.69e-03)      -ves (3.29e-03)       /manage (3.51e-03)   
9      /customer (4.72e-03)   personn (2.89e-03)  /environment (3.43e-03)   
10  /preferences (4.28e-03)         ť (2.83e-03)  /preferences (3.18e-03)   
11       /shared (3.43e-03)     probs (2.76e-03)          .Abs (2.97e-03)   
12       /entity (3.23e-03)      elig (2.51e-03)       perpetr (2.67e-03)   
13       /dialog (3.23e-03)    ):\n\n (2.38e-03)             — (2.64e-03)   
14      /account (3.12e-03)      roku (2.33e-03)             / (2.50e-03)   
15      libertin (3.04e-03)       )": (2.27e-03)       /dialog (2.38e-03)   
16       /layout (2.91e-03)     lesen (2.21e-03)     /customer (2.37e-03)   
17          .Try (2.87e-03)  ,,,,,,,, (2.20e-03)        /basic (2.27e-03)   
18         .Take (2.71e-03)       (us (2.12e-03)       /entity (2.19e-03)   
19      /effects (2.71e-03)    spiele (2.11e-03)   /categories (2.19e-03)   

                                                                           \
                  ft_inv                      diff               diff_inv   
0           .vn (0.0164)                — (0.1283)           � (8.56e-03)   
1     /Register (0.0137)                ■ (0.0149)            (5.95e-03)   
2           -ie (0.0102)              ..\ (0.0108)       début (4.10e-03)   
3    misunder (8.85e-03)              ■ (8.69e-03)       poste (3.44e-03)   
4      testim (6.21e-03)  ' \n\n\n\n\n' (7.64e-03)         keh (2.28e-03)   
5       probs (5.71e-03)       ———————— (6.97e-03)         oad (2.11e-03)   
6        sagt (5.21e-03)          ' \n' (5.39e-03)        )`\n (2.06e-03)   
7       scrut (5.10e-03)        />.\n\n (5.28e-03)        jian (1.91e-03)   
8        elig (4.94e-03)  <|endoftext|> (4.77e-03)          aż (1.84e-03)   
9       asign (4.68e-03)        ' \n\n' (4.07e-03)         fic (1.80e-03)   
10         że (4.47e-03)           pond (3.96e-03)  /providers (1.73e-03)   
11    personn (3.72e-03)             —— (3.65e-03)      anmeld (1.72e-03)   
12       helf (3.70e-03)           agos (3.58e-03)          ół (1.71e-03)   
13   ,,,,,,,, (3.64e-03)           ———— (2.78e-03)         neh (1.57e-03)   
14       -ves (3.59e-03)          resse (2.16e-03)         ﻿\n (1.50e-03)   
15      zeigt (3.32e-03)             ,— (2.14e-03)      .Entry (1.47e-03)   
16   protagon (2.95e-03)            <\/ (1.97e-03)           ъ (1.45e-03)   
17      lesen (2.61e-03)            Fil (1.91e-03)        über (1.42e-03)   
18          ť (2.50e-03)         Cancel (1.90e-03)         .iv (1.41e-03)   
19     ):\n\n (2.29e-03)           ,... (1.87e-03)        )!\n (1.39e-03)   

             layer_14                                           \
                 base              base_inv                 ft   
0          , (0.7987)     contador (0.9618)         , (0.6939)   
1        ' ' (0.1049)    kontrol (3.12e-03)       ' ' (0.2344)   
2        the (0.0467)   karakter (2.39e-03)       and (0.0256)   
3        and (0.0320)       rekl (1.60e-03)       the (0.0251)   
4       in (6.38e-03)         bö (1.54e-03)      in (6.73e-03)   
5        ( (3.66e-03)       zoek (1.20e-03)       ( (4.81e-03)   
6       's (2.61e-03)     testim (9.38e-04)       . (4.40e-03)   
7        a (1.87e-03)    Produto (9.

## 2. PatchScope Analysis

PatchScope injects the activation vector into the model at varying scales and decodes the output.  
Unlike logit lens, there are no inverse variants -- only `base`, `ft`, and `diff`.  
Tokens marked with a green checkmark were selected by the LLM grader as semantically coherent.

### 2A. Single Position

Shows tokens at the best scale found by the auto patch scope search.  
Format: `token (prob)` with `\u2705` if in `selected_tokens`

In [7]:
PS_VARIANTS = [("base", "base_"), ("ft", "ft_"), ("diff", "")]


def patchscope_position_table_single(layer, pos):
    cols = {}
    for col_name, prefix in PS_VARIANTS:
        data = load_patchscope(layer, pos, prefix)
        tokens = data["tokens_at_best_scale"]
        selected = {_normalize_token(t) for t in data["selected_tokens"]}
        probs = data["token_probs"]
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(p)})"
            + (" \u2705" if _normalize_token(t) in selected else "")
            for t, p in zip(tokens, probs)
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_position_table(pos):
    dfs = []
    for layer in LAYERS:
        df = patchscope_position_table_single(layer, pos)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


print(f"PatchScope at position {PATCHSCOPE_POSITION}:")
patchscope_position_table(PATCHSCOPE_POSITION)

PatchScope at position -1:


layer_7                                                   \
                      base                         ft                  diff   
0             The (0.1598)             oller (0.0221)          ' ' (0.0401)   
1           There (0.0400)      concrete (5.40e-03) ✅          ... (0.0381)   
2              As (0.0396)         Buccane (5.14e-03)      thank (0.0273) ✅   
3          When (0.0212) ✅     generally (4.38e-03) ✅           42 (0.0219)   
4            If (0.0168) ✅       current (3.67e-03) ✅    message (0.0189) ✅   
5              It (0.0165)            fern (3.64e-03)            n (0.0147)   
6         Given (0.0163) ✅            .Abs (3.42e-03)      Thank (0.0137) ✅   
7             For (0.0143)       studies (2.77e-03) ✅           na (0.0122)   
8              To (0.0137)     according (2.69e-03) ✅    Thank (8.92e-03) ✅   
9            This (0.0126)         would (2.66e-03) ✅   thanks (8.00e-03) ✅   
10              A (0.0119)            iger (2.59e-03)         th (7.40e-03)   
11             In (0.0119)          .There (2.34e-03)          g (6.30e-03)   
12            You (0.0105)          wouldn (2.22e-03)          n (6.00e-03)   
13        While (0.0104) ✅       depends (2.00e-03) ✅    thank (5.84e-03) ✅   
14     Having (9.99e-03) ✅     depending (1.98e-03) ✅     target (5.83e-03)   
15   Although (9.30e-03) ✅             cly (1.87e-03)     anna (5.79e-03) ✅   
16          Let (9.01e-03)    engagement (1.73e-03) ✅         is (5.69e-03)   
17    Several (8.91e-03) ✅             riv (1.65e-03)         if (5.59e-03)   
18  According (8.04e-03) ✅   sovereignty (1.64e-03) ✅        day (5.53e-03)   
19    Despite (7.63e-03) ✅      contrary (1.63e-03) ✅         ow (5.13e-03)   

                layer_14                                                      \
                    base                          ft                    diff   
0            To (0.7370)                The (0.1849)        Thank (0.0341) ✅   
1           ### (0.1230)                 As (0.1699)              1 (0.0225)   
2            ** (0.0658)                 To (0.1562)         Dear (0.0118) ✅   
3         Let (0.0535) ✅                  I (0.1436)    mission (9.56e-03) ✅   
4           The (0.0147)             Sure (0.1165) ✅      Sorry (8.45e-03) ✅   
5   Certainly (1.37e-03)                  1 (0.1029)      Hello (7.93e-03) ✅   
6        Sure (9.82e-04)             Here (0.0203) ✅          zou (7.46e-03)   
7          In (7.31e-04)        Certainly (0.0172) ✅        Based (6.72e-03)   
8          ## (6.87e-04)            Thank (0.0109) ✅     Please (5.93e-03) ✅   
9     First (2.38e-04) ✅            Based (9.58e-03)       mail (5.23e-03) ✅   
10    Given (2.38e-04) ✅               In (7.46e-03)        thank (4.61e-03)   
11         We (1.30e-04)               It (7.16e-03)     Notify (4.43e-03) ✅   
12          1 (1.27e-04)            There (5.13e-03)      Great (4.25e-03) ✅   
13    Alright (1.10e-04)          Great (4.72e-03) ✅            I (3.52e-03)   
14     This (1.05e-04) ✅  Unfortunately (3.99e-03) ✅          >\n (3.31e-03)   
15     Here (9.47e-05) ✅            Yes (3.51e-03) ✅          mer (3.06e-03)   
16       #### (8.90e-05)             This (3.10e-03)         Line (2.80e-03)   
17        ``` (8.22e-05)              Let (2.75e-03)     ulfilled (2.63e-03)   
18        For (6.93e-05)              For (2.33e-03)  Regarding (2.63e-03) ✅   
19         As (6.64e-05)          Sorry (1.30e-03) ✅      However (2.63e-03)   

                layer_15                                                  
                    base                        ft                  diff  
0            To (0.4434)                I (0.2168)            I (0.1543)  
1            ** (0.2695)              The (0.1914)        Thank (0.1357)  
2           ### (0.2373)               As (0.1914)          >\n (0.0938)  
3         Let (0.0250) ✅               To (0.1162)       Dear (0.0442) ✅  
4           The (0.0172)           Sure (0.1025) ✅           As (0.03

### 2B. Aggregated Across All PatchScope Positions

Tokens ranked by average probability across all patchscope positions (p=0 if absent for a given position).  
Green checkmark if the token was in `selected_tokens` for **any** position.  
Format: `token (avg_prob)`

In [8]:
def patchscope_aggregated_single(layer):
    ps_positions = discover_patchscope_positions(layer)
    n_ps = len(ps_positions)

    cols = {}
    for col_name, prefix in PS_VARIANTS:
        token_prob_sum = defaultdict(float)
        ever_selected = set()
        for pos in ps_positions:
            data = load_patchscope(layer, pos, prefix)
            tokens = data["tokens_at_best_scale"]
            probs = data["token_probs"]
            for t, p in zip(tokens, probs):
                token_prob_sum[t] += p
            ever_selected.update(_normalize_token(t) for t in data["selected_tokens"])

        token_avg = {t: s / n_ps for t, s in token_prob_sum.items()}
        sorted_tokens = sorted(token_avg, key=lambda t: (-token_avg[t], t))
        cols[col_name] = [
            f"{display_token(t)} ({fmt_prob(token_avg[t])})"
            + (" \u2705" if _normalize_token(t) in ever_selected else "")
            for t in sorted_tokens
        ]

    max_len = max(len(v) for v in cols.values())
    for k in cols:
        cols[k] += [""] * (max_len - len(cols[k]))
    return pd.DataFrame(cols)


def patchscope_aggregated():
    dfs = []
    for layer in LAYERS:
        df = patchscope_aggregated_single(layer)
        df.columns = pd.MultiIndex.from_product([[f"layer_{layer}"], df.columns])
        dfs.append(df)
    return concat_layer_dfs(dfs)


ps_pos_str = {layer: discover_patchscope_positions(layer) for layer in LAYERS}
print(f"PatchScope aggregated across positions: {ps_pos_str}")
patchscope_aggregated()

PatchScope aggregated across positions: {7: [0, 1, 2, 3, 4, 5], 14: [0, 1, 2, 3, 4, 5], 15: [0, 1, 2, 3, 4, 5]}


layer_7                             \
                         base                         ft   
0                 -> (0.0697)         problem (0.0659) ✅   
1          problem (0.0386) ✅               the (0.0186)   
2              :\n\n (0.0347)        /problem (0.0183) ✅   
3                 's (0.0257)                 , (0.0161)   
4                the (0.0249)                's (0.0122)   
5            solve (0.0241) ✅       /problems (0.0113) ✅   
6             '\n\n' (0.0193)      question (9.22e-03) ✅   
7         /problem (0.0184) ✅     /entities (8.73e-03) ✅   
8        /entities (0.0132) ✅        puzzle (6.60e-03) ✅   
9                :\n (0.0107)           :\n\n (6.57e-03)   
10              '\n' (0.0105)             :\n (5.83e-03)   
11             you (9.28e-03)       /manage (5.82e-03) ✅   
12     /problems (9.05e-03) ✅         solve (5.27e-03) ✅   
13     statement (8.05e-03) ✅               I (4.97e-03)   
14            this (7.93e-03)    understand (4.79e-03) ✅   
15               , (6.75e-03)         start (3.97e-03) ✅   
16    understand (6.46e-03) ✅            '\n' (3.81e-03)   
17      question (5.89e-03) ✅      problems (3.53e-03) ✅   
18           seems (5.22e-03)              -> (3.18e-03)   
19       /manage (4.59e-03) ✅             you (3.01e-03)   
20       analyze (4.23e-03) ✅     summarize (2.86e-03) ✅   
21       address (3.70e-03) ✅       clarify (2.66e-03) ✅   
22        .Today (3.63e-03) ✅            this (2.60e-03)   
23          math (3.42e-03) ✅            John (2.54e-03)   
24        solves (3.06e-03) ✅       /global (2.22e-03) ✅   
25       /global (2.86e-03) ✅      /logging (2.19e-03) ✅   
26        puzzle (2.84e-03) ✅              in (2.00e-03)   
27          task (2.79e-03) ✅               : (1.97e-03)   
28            your (2.75e-03)         begin (1.86e-03) ✅   
29               : (2.71e-03)  /environment (1.84e-03) ✅   
30              is (2.60e-03)              we (1.83e-03)   
31  /preferences (2.59e-03) ✅      /account (1.78e-03) ✅   
32        tackle (2.52e-03) ✅            bear (1.53e-03)   
33      problems (2.44e-03) ✅       /shared (1.52e-03) ✅   
34              ’s (2.35e-03)           scarc (1.50e-03)   
35          step (2.34e-03) ✅       puzzles (1.47e-03) ✅   
36         break (1.99e-03) ✅          math (1.42e-03) ✅   
37          /job (1.98e-03) ✅          .Today (1.41e-03)   
38     /provider (1.89e-03) ✅       analyze (1.32e-03) ✅   
39       /layout (1.87e-03) ✅               a (1.32e-03)   
40      /logging (1.83e-03) ✅              to (1.25e-03)   
41      solution (1.76e-03) ✅            with (1.20e-03)   
42       /crypto (1.72e-03) ✅       address (1.20e-03) ✅   
43        decode (1.42e-03) ✅        answer (1.18e-03) ✅   
44              we (1.38e-03)     /provider (1.17e-03) ✅   
45       /engine (1.36e-03) ✅       proceed (1.14e-03) ✅   
46       puzzles (1.18e-03) ✅  /preferences (1.07e-03) ✅   
47   /connection (1.16e-03) ✅             for (1.02e-03)   
48      /effects (1.07e-03) ✅          .Round (1.02e-03)   
49        begins (1.06e-03) ✅   /connection (1.02e-03) ✅   
50      involves (1.02e-03) ✅          find (1.02e-03) ✅   
51       /entity (1.01e-03) ✅     questions (9.82e-04) ✅   
52        /legal (1.00e-03) ✅          /inc (9.43e-04) ✅   
53          /man (9.33e-04) ✅       /layout (9.36e-04) ✅   
54      exercise (8.90e-04) ✅        /basic (9.27e-04) ✅   
55          word (8.82e-04) ✅          task (9.12e-04) ✅   
56        answer (8.15e-04) ✅        /legal (8.37e-04) ✅   
57       /object (7.82e-04) ✅        concerns (4.54e-04)   
58      requires (7.65e-04) ✅          '\n\n' (4.25e-04)   
59       example (7.49e-04) ✅         /apis (4.02e-04) ✅   
60        prompt (7.06e-04) ✅    /functions (3.92e-04) ✅   
61             /pr (6.93e-04)               . (3.87e-04)   
62  /environment (6.75e-04) ✅           /disc (3.69e-04)   
63      /testing (6.45e-04) ✅       /events (3.62e-04) ✅   
64          /reg (6.16e-04) ✅     /activity (3.36e-04) ✅   
65

## 3. Diff Logit Lens Across Positions

Shows only the **diff** variant of the logit lens for selected positions across all layers.
Format: `token (softmax_prob)`

In [9]:
DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3, 10, 50, 100]


def logit_lens_diff_positions_table():
    """Show diff logit lens across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in DIFF_POSITIONS:
            prefix, pi, ii = LL_VARIANTS["diff"]
            data = load_logit_lens(layer, pos, prefix)
            tokens = decode_tokens(data[ii])
            probs = data[pi].tolist()
            col = [f"{display_token(t)} ({fmt_prob(p)})" for t, p in zip(tokens, probs)]
            if LOGIT_LENS_MAX_ROWS is not None:
                col = col[:LOGIT_LENS_MAX_ROWS]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame(col_data)
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"Logit lens DIFF across positions: {DIFF_POSITIONS}")
logit_lens_diff_positions_table()

Logit lens DIFF across positions: [-3, -1, 0, 1, 2, 3, 10, 50, 100]


layer_7                                                  \
                     pos_-3                  pos_-1                   pos_0   
0           bone (9.64e-03)            ach (0.0129)           .. (7.93e-03)   
1            ikh (5.31e-03)          ' \n' (0.0118)          mon (6.77e-03)   
2            Alg (5.16e-03)        domin (7.17e-03)        Olymp (4.97e-03)   
3            lio (4.15e-03)          mon (7.17e-03)     Archives (4.97e-03)   
4         shapes (3.89e-03)        Abyss (5.95e-03)    statement (4.52e-03)   
5       Telegram (3.66e-03)            次 (3.72e-03)        arton (4.12e-03)   
6           Tear (2.76e-03)            @ (3.60e-03)        ached (4.00e-03)   
7          chein (2.76e-03)            ( (3.17e-03)          Ana (3.86e-03)   
8            ную (2.67e-03)            � (2.99e-03)     lections (3.63e-03)   
9          också (2.67e-03)   influences (2.72e-03)         ster (3.42e-03)   
10          gaat (2.59e-03)            O (2.64e-03)          sur (3.20e-03)   
11          Sche (2.52e-03)       master (2.64e-03)       fabric (3.01e-03)   
12        inject (2.52e-03)         Gold (2.55e-03)          ..\ (2.91e-03)   
13          chai (2.44e-03)            d (2.55e-03)         Hipp (2.91e-03)   
14           iid (2.37e-03)      supreme (2.47e-03)         olon (2.91e-03)   
15       Anthrop (2.29e-03)      pressed (2.47e-03)          abh (2.91e-03)   
16   Legislation (2.21e-03)     imestamp (2.29e-03)   statements (2.82e-03)   
17     continuum (2.21e-03)        -eyed (2.26e-03)         iphy (2.58e-03)   
18         lucky (2.15e-03)        oller (2.01e-03)          den (2.35e-03)   
19        oplast (2.15e-03)          sea (1.95e-03)            — (2.27e-03)   

                                                        \
                       pos_1                     pos_2   
0                 — (0.0247)                — (0.1396)   
1          Contract (0.0179)         Contract (0.0243)   
2        caratter (6.04e-03)      contracts (8.36e-03)   
3       contracts (5.86e-03)          ' \n' (7.87e-03)   
4        lections (5.16e-03)       contract (3.97e-03)   
5               _ (4.43e-03)           auté (3.97e-03)   
6         qualche (4.03e-03)          Dimit (3.60e-03)   
7           opper (3.66e-03)           able (3.08e-03)   
8             den (3.33e-03)       contract (2.99e-03)   
9           ' \n' (3.04e-03)           whom (2.90e-03)   
10            ..\ (2.94e-03)  ' \n\n\n\n\n' (2.90e-03)   
11       Archives (2.44e-03)            ..\ (2.56e-03)   
12            mon (2.37e-03)          abled (2.33e-03)   
13         storia (2.29e-03)           pond (2.26e-03)   
14          ensed (2.21e-03)        qualche (2.26e-03)   
15          igned (2.09e-03)            PPP (2.26e-03)   
16          enson (2.01e-03)            PHA (2.00e-03)   
17   publications (2.01e-03)          marty (1.87e-03)   
18         martyr (1.90e-03)          pueda (1.82e-03)   
19            bil (1.90e-03)          bonds (1.82e-03)   

                                                                              \
                       pos_3                  pos_10                  pos_50   
0                 — (0.2598)              — (0.2090)              — (0.1631)   
1            auté (7.11e-03)          ' \n' (0.0142)  <|endoftext|> (0.0162)   
2           ' \n' (6.29e-03)      ' \n\n' (9.46e-03)              ■ (0.0118)   
3        Contract (5.22e-03)     lections (4.33e-03)     ———————— (7.87e-03)   
4           Dimit (4.06e-03)            ■ (4.21e-03)          ..\ (7.63e-03)   
5       contracts (3.81e-03)         fran (2.88e-03)            ■ (6.74e-03)   
6             ..\ (2.70e-03)  ' \n\n\n\n' (2.47e-03)      />.\n\n (5.25e-03)   
7        contract (2.53e-03)        Dimit (2.47e-03)         pond (4.94e-03)   
8            pond (2.17e-03)     ulfilled (2.40e-03)         agos (4.79e-03)   
9            whom (1.91e-03)            ■ (2.18e-03)        resse (4.36e-03)   
10          bonds (1.75e-03)        beats (2

## 4. Diff PatchScope Across Positions

Shows only the **diff** variant of PatchScope for selected positions across all layers.
Format: `token (prob)` with `✅` if in `selected_tokens`

In [10]:
PS_DIFF_POSITIONS = [-3, -1, 0, 1, 2, 3]


def patchscope_diff_positions_table():
    """Show diff patchscope across multiple positions for all layers."""
    dfs = []
    for layer in LAYERS:
        col_data = {}
        for pos in PS_DIFF_POSITIONS:
            try:
                data = load_patchscope(layer, pos, prefix="")
            except FileNotFoundError:
                col_data[f"pos_{pos}"] = ["(not available)"]
                continue
            tokens = data["tokens_at_best_scale"]
            selected = {_normalize_token(t) for t in data["selected_tokens"]}
            probs = data["token_probs"]
            col = [
                f"{display_token(t)} ({fmt_prob(p)})"
                + (" ✅" if _normalize_token(t) in selected else "")
                for t, p in zip(tokens, probs)
            ]
            col_data[f"pos_{pos}"] = col
        layer_df = pd.DataFrame({k: pd.Series(v) for k, v in col_data.items()}).fillna(
            ""
        )
        layer_df.columns = pd.MultiIndex.from_product(
            [[f"layer_{layer}"], layer_df.columns]
        )
        dfs.append(layer_df)
    return concat_layer_dfs(dfs)


print(f"PatchScope DIFF across positions: {PS_DIFF_POSITIONS}")
patchscope_diff_positions_table()

PatchScope DIFF across positions: [-3, -1, 0, 1, 2, 3]


layer_7                                                  \
                      pos_-3                pos_-1                     pos_0   
0            bone (7.91e-03)          ' ' (0.0401)           olon (7.40e-03)   
1             Alg (6.99e-03)          ... (0.0381)            mon (7.09e-03)   
2            enty (6.16e-03)      thank (0.0273) ✅            Ana (6.21e-03)   
3         press (3.93e-03) ✅           42 (0.0219)    statement (5.19e-03) ✅   
4            Tear (3.93e-03)    message (0.0189) ✅     Archives (5.08e-03) ✅   
5             ikh (3.91e-03)            n (0.0147)          Olymp (4.82e-03)   
6        Telegram (3.77e-03)      Thank (0.0137) ✅           ster (4.17e-03)   
7            Sche (3.55e-03)           na (0.0122)         fabric (4.14e-03)   
8             iid (3.41e-03)    Thank (8.92e-03) ✅          ached (3.96e-03)   
9             alg (3.30e-03)   thanks (8.00e-03) ✅            abh (3.38e-03)   
10            ную (3.20e-03)         th (7.40e-03)   statements (3.21e-03) ✅   
11       shapes (3.04e-03) ✅          g (6.30e-03)          iston (3.08e-03)   
12            lio (2.98e-03)          n (6.00e-03)         Blanch (3.05e-03)   
13          uture (2.98e-03)    thank (5.84e-03) ✅           Hipp (3.02e-03)   
14   Hutchinson (2.66e-03) ✅     target (5.83e-03)             .. (2.99e-03)   
15        Posting (2.65e-03)     anna (5.79e-03) ✅            Ana (2.99e-03)   
16    continuum (2.52e-03) ✅         is (5.69e-03)        qualche (2.69e-03)   
17           gaat (2.47e-03)         if (5.59e-03)    contracts (2.43e-03) ✅   
18     readings (2.39e-03) ✅        day (5.53e-03)            Haz (2.40e-03)   
19            Alg (2.15e-03)         ow (5.13e-03)     Contract (2.33e-03) ✅   

                                                                          \
                   pos_1                     pos_2                 pos_3   
0            -> (0.1737)              ' ' (0.0579)           -> (0.1787)   
1           ' ' (0.0247)               -> (0.0572)       blue (0.0687) ✅   
2           ann (0.0163)             anna (0.0423)          ' ' (0.0250)   
3             0 (0.0134)           blue (0.0380) ✅         anna (0.0230)   
4          anna (0.0119)          green (0.0335) ✅      green (0.0223) ✅   
5        blue (0.0112) ✅              man (0.0256)          ann (0.0148)   
6           man (0.0105)              ann (0.0156)          113 (0.0132)   
7           ( (8.98e-03)          green (0.0102) ✅           -> (0.0129)   
8           1 (8.09e-03)          red (9.67e-03) ✅        ana (9.85e-03)   
9     green (7.22e-03) ✅  <|endoftext|> (8.61e-03)       '\n' (9.77e-03)   
10       Anna (6.76e-03)            man (7.89e-03)         is (6.61e-03)   
11      red (6.45e-03) ✅            men (6.89e-03)        ann (6.41e-03)   
12   yellow (5.28e-03) ✅           '\n' (6.14e-03)      red (6.40e-03) ✅   
13         -> (5.00e-03)            ana (5.92e-03)          0 (5.61e-03)   
14       '\n' (4.74e-03)            ann (5.54e-03)         an (5.24e-03)   
15        men (4.64e-03)            113 (5.34e-03)      hello (4.92e-03)   
16         is (4.62e-03)           same (5.17e-03)          [ (4.83e-03)   
17   orange (4.10e-03) ✅              a (5.12e-03)          1 (4.73e-03)   
18          " (4.08e-03)             is (5.03e-03)   yellow (4.67e-03) ✅   
19      hello (3.97e-03)       yellow (4.55e-03) ✅          a (4.66e-03)   

              layer_14                          \
                pos_-3                  pos_-1   
0           | (0.9609)        Thank (0.0341) ✅   
1           > (0.0176)              1 (0.0225)   
2        >> (2.70e-03)         Dear (0.0118) ✅   
3        >| (1.54e-03)    mission (9.56e-03) ✅   
4         » (1.27e-03)      Sorry (8.45e-03) ✅   
5         ＞ (1.04e-03)      Hello (7.93e-03) ✅   
6        |i (4.14e-04)          zou (7.46e-03)   
7        >( (3.43e-04)        Based (6.72e-03)   
8   |string (3.03e-04)     Please (5.93e-03) ✅   
9        |( (2.84e-04)       mail (5.